In [1]:
import pandas as pd

print("Loading files...")
df_a = pd.read_excel('CSV 1 Final.xlsx')
df_b = pd.read_excel('CSV 2 Final.xlsx')

# 1. Fix any hidden spaces in column names from Excel
df_a.columns = df_a.columns.astype(str).str.strip()
df_b.columns = df_b.columns.astype(str).str.strip()

# 2. Get lowercased column names to do a safe check
cols_a = [str(c).lower() for c in df_a.columns]
cols_b = [str(c).lower() for c in df_b.columns]

# 3. Auto-detect which file is Weather (has year/month) and which is AQI (has datetime)
if 'year' in cols_a:
    weather_df = df_a
    aqi_df = df_b
else:
    weather_df = df_b
    aqi_df = df_a

# 4. Safely drop 'Unnamed' junk columns
weather_df = weather_df.loc[:, ~weather_df.columns.str.contains('^Unnamed')]
aqi_df = aqi_df.loc[:, ~aqi_df.columns.str.contains('^Unnamed')]

# 5. Rename Year/Month/Day/Hour to strictly lowercase so pd.to_datetime works perfectly
rename_dict = {col: col.lower() for col in weather_df.columns if col.lower() in ['year', 'month', 'day', 'hour']}
weather_df = weather_df.rename(columns=rename_dict)

# 6. Create datetime in weather data
weather_df['datetime'] = pd.to_datetime(weather_df[['year', 'month', 'day', 'hour']])

# 7. Find the exact name of the datetime column in AQI data and convert it
aqi_dt_col = [c for c in aqi_df.columns if c.lower() == 'datetime'][0]
aqi_df = aqi_df.rename(columns={aqi_dt_col: 'datetime'})
aqi_df['datetime'] = pd.to_datetime(aqi_df['datetime'])

# 8. Merge, fill missing, and save
merged_df = pd.merge(weather_df, aqi_df, on='datetime', how='inner')
merged_df = merged_df.ffill()
merged_df.to_csv('ME228_Final_Merged_Dataset.csv', index=False)

print("Merge complete! Total rows:", len(merged_df))

Loading files...
Merge complete! Total rows: 29011
